In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [2]:
raw = np.load("/content/PEMS08.npz")
data = raw["data"].astype(np.float32)

# Use traffic flow channel
data = data[:, :, 0]   # (time, sensors)

SENSORS = data.shape[1]
print("Data shape:", data.shape)

Data shape: (17856, 170)


In [3]:
scaler = MinMaxScaler()
data = scaler.fit_transform(data)

In [4]:
SEQ_LEN = 96
PRED_LEN = 12

def create_sequences(data):
    X, Y = [], []
    for i in range(len(data) - SEQ_LEN - PRED_LEN):
        X.append(data[i:i+SEQ_LEN])
        Y.append(data[i+SEQ_LEN:i+SEQ_LEN+PRED_LEN])
    return np.array(X), np.array(Y)

X, Y = create_sequences(data)

print(X.shape, Y.shape)

(17748, 96, 170) (17748, 12, 170)


In [5]:
class TrafficDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X).float()
        self.Y = torch.tensor(Y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

dataset = TrafficDataset(X, Y)

split = int(0.8 * len(dataset))
train_set = torch.utils.data.Subset(dataset, range(split))
val_set   = torch.utils.data.Subset(dataset, range(split, len(dataset)))

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=32)

In [6]:
class CNN_LSTM_LLM(nn.Module):
    def __init__(self, sensors, pred_len):
        super().__init__()

        hidden = 128

        # Temporal CNN (1D)
        self.cnn = nn.Sequential(
            nn.Conv1d(sensors, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        # LSTM
        self.lstm = nn.LSTM(
            128,
            hidden,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )

        # Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=8,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=3
        )

        self.fc = nn.Linear(hidden, sensors * pred_len)

        self.pred_len = pred_len
        self.sensors = sensors

    def forward(self, x):
        # x: (B, T, S)

        x = x.permute(0, 2, 1)  # (B, S, T)

        x = self.cnn(x)         # (B, 128, T)

        x = x.permute(0, 2, 1)  # (B, T, 128)

        x, _ = self.lstm(x)

        x = self.transformer(x)

        x = x[:, -1, :]         # last time step

        out = self.fc(x)

        return out.view(-1, self.pred_len, self.sensors)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_LSTM_LLM(SENSORS, PRED_LEN).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=3,
    factor=0.5
)

loss_fn = nn.HuberLoss()

In [8]:
best_val = 1e9
patience = 8
counter = 0

for epoch in range(60):

    model.train()
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        pred = model(x)
        loss = loss_fn(pred, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            val_loss += loss_fn(pred, y).item()

    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    print(epoch, train_loss, val_loss)

    if val_loss < best_val:
        best_val = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

0 0.017228172644175723 0.004908465121155353
1 0.0042948712367220435 0.004058364815989023
2 0.003831414385228879 0.003835084967143196
3 0.003487829308797446 0.0034362303133978435
4 0.0031654305980532422 0.0030687549990509543
5 0.0029015301801222334 0.003032814537233732
6 0.002735470522295784 0.0028053756020879287
7 0.002569006179901981 0.002621865690778102
8 0.0024476135320025956 0.0026352258791375134
9 0.002371941628102381 0.0025904558154309656
10 0.0022946713200711587 0.002541860729678291
11 0.0022108726505492187 0.002535415847412403
12 0.0021465680069554394 0.0023893321487268113
13 0.002072219972326714 0.002371292401614876
14 0.002007619893128002 0.0023471136584768894
15 0.001946682078703013 0.0023388180686139946
16 0.0018891228097050476 0.002248977123222708
17 0.0018253907501521345 0.0023177622131550233
18 0.0017860426175773042 0.0022033713550856423
19 0.001739683785729597 0.002216471103754163
20 0.0016988833138073148 0.002174776493202526
21 0.001663668191811355 0.002122166685850217

In [9]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

preds, actual = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        p = model(x).cpu().numpy()
        preds.append(p)
        actual.append(y.numpy())

preds = np.concatenate(preds)
actual = np.concatenate(actual)

preds_denorm = scaler.inverse_transform(
    preds.reshape(-1, preds.shape[-1])
).reshape(preds.shape)

actual_denorm = scaler.inverse_transform(
    actual.reshape(-1, actual.shape[-1])
).reshape(actual.shape)

mae  = np.mean(np.abs(preds_denorm - actual_denorm))
rmse = np.sqrt(np.mean((preds_denorm - actual_denorm)**2))

mask = actual_denorm > 10
mape = np.mean(
    np.abs((actual_denorm[mask] - preds_denorm[mask]) /
           actual_denorm[mask])
) * 100

print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

MAE : 19.359835
RMSE: 30.747265
MAPE: 11.049533
